# NLP Model Evaluation vs Baseline Paper Results

This notebook evaluates the NLP-based BFRB detection model against the baseline results reported in:
> Zhang, Ryoo, Mukherjee (2025), *Detection of Body Focused Repetitive Behaviors using Deep Learning*.

**Metrics used:**
- Binary F1-score (BFRB vs non-target)
- Macro-averaged F1-score across 8 BFRB gesture classes

## 1. Imports and Configuration

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay

import warnings
warnings.filterwarnings('ignore')

# Paths — outputs written by nlp_model.ipynb
OUTPUTS_DIR = '../models_artifacts/outputs/'
METADATA_PATH = '../models_artifacts/metadata/class_mapping.json'
NLP_RESULTS_PATH = os.path.join(OUTPUTS_DIR, 'nlp_model_results.json')
NLP_LOGITS_PATH  = os.path.join(OUTPUTS_DIR, 'logits_val_nlp_model.npy')
PLOTS_DIR = '../plots/'
os.makedirs(PLOTS_DIR, exist_ok=True)

## 2. Load NLP Model Results

> **Prerequisite:** Run `nlp_model.ipynb` first to generate `nlp_model_results.json` and `logits_val_nlp_model.npy`.

In [ ]:
with open(NLP_RESULTS_PATH, 'r') as f:
    nlp_results = json.load(f)

nlp_logits = np.load(NLP_LOGITS_PATH)

print('NLP Model Results:')
print(json.dumps(nlp_results, indent=2))
print(f'\nLogits shape: {nlp_logits.shape}')

## 3. Baseline Paper Results (Table from Zhang et al., 2025)

In [ ]:
# Results as reported in Zhang, Ryoo, Mukherjee (2025)
baseline_results = {
    'FFT-MLP (IMU+THM+TOF)': {'binary_f1': 0.97, 'macro_f1': 0.73},
    'FFT-MLP (IMU+THM)':     {'binary_f1': 0.95, 'macro_f1': 0.68},
    'CNN-BiLSTM (TOF)':      {'binary_f1': 0.91, 'macro_f1': 0.61},
    'Late Fusion Ensemble':  {'binary_f1': 0.97, 'macro_f1': 0.75},
    'Intermediate Fusion':   {'binary_f1': 0.96, 'macro_f1': 0.72},
    'FFT-Random Forest':     {'binary_f1': 0.94, 'macro_f1': 0.65},
}

baseline_df = pd.DataFrame(baseline_results).T
baseline_df.index.name = 'Model'
print('Baseline Paper Results:')
baseline_df

## 4. Comparison Table: NLP Model vs Baselines

In [ ]:
comparison = baseline_df.copy()
comparison.loc['NLP-TF-IDF-LR (Ours)'] = [
    nlp_results['binary_f1'],
    nlp_results['macro_f1']
]

comparison = comparison.sort_values('macro_f1', ascending=False)
comparison.columns = ['Binary F1', 'Macro F1']
comparison = comparison.round(4)

print('Full Comparison Table:')
comparison

## 5. Binary F1 Comparison Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['steelblue' if idx != 'NLP-TF-IDF-LR (Ours)' else 'darkorange'
          for idx in comparison.index]

comparison['Binary F1'].plot(kind='barh', ax=ax, color=colors)
ax.set_xlabel('Binary F1-Score')
ax.set_title('Binary F1-Score: NLP Model vs Paper Baselines')
ax.axvline(x=nlp_results['binary_f1'], color='darkorange', linestyle='--', linewidth=1.2, label='NLP model')
ax.legend()
ax.set_xlim(0, 1.05)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'eval_binary_f1_comparison.png'), dpi=150)
plt.show()
print('Saved: plots/eval_binary_f1_comparison.png')

## 6. Macro F1 Comparison Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
comparison['Macro F1'].plot(kind='barh', ax=ax, color=colors)
ax.set_xlabel('Macro-Averaged F1-Score')
ax.set_title('Macro F1-Score: NLP Model vs Paper Baselines')
ax.axvline(x=nlp_results['macro_f1'], color='darkorange', linestyle='--', linewidth=1.2, label='NLP model')
ax.legend()
ax.set_xlim(0, 1.05)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'eval_macro_f1_comparison.png'), dpi=150)
plt.show()
print('Saved: plots/eval_macro_f1_comparison.png')

## 7. Save Comparison Table to CSV

In [ ]:
comparison_out_path = os.path.join(OUTPUTS_DIR, 'evaluation_comparison.csv')
comparison.to_csv(comparison_out_path)
print(f'Comparison table saved to: {comparison_out_path}')
print('\n--- This file is the input for plot_generation.ipynb ---')

## 8. Summary

In [ ]:
best_baseline_macro  = baseline_df['macro_f1'].max()
best_baseline_binary = baseline_df['binary_f1'].max()

print('=== Evaluation Summary ===')
print(f"NLP Model Binary F1 : {nlp_results['binary_f1']:.4f}  (best baseline: {best_baseline_binary:.4f})")
print(f"NLP Model Macro F1  : {nlp_results['macro_f1']:.4f}  (best baseline: {best_baseline_macro:.4f})")
delta_macro = nlp_results['macro_f1'] - best_baseline_macro
print(f"\nMacro F1 gap vs best baseline: {delta_macro:+.4f}")